# Breast Cancer Prediction — Benign vs. Malignant Classification

**Author:** John | Medical Laboratory Scientist → ML/Data Science
**Dataset:** [Breast Cancer Wisconsin (Diagnostic) Data Set — Kaggle](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data)
**Repo:** see `README.md` for full write-up, results, and reproduction steps.

This notebook trains and compares 7 classification algorithms (baseline + hyperparameter-tuned versions)
on cell-nuclei features derived from digitized breast mass biopsies, to classify tumors as
**benign (0)** or **malignant (1)**. Recall on the malignant class is treated as the primary
evaluation metric, since a missed malignant case (false negative) is the costliest error in this
clinical context.

---


# *Breast Cancer Prediction*

**Problem Statement**: To create a predictive model that can accurately classify breast
cancer cases as benign or malignant based on a set of relevant features. By leveraging
historical data and applying machine learning techniques, we aim to develop a reliable tool for
assisting medical professionals in diagnosing breast cancer.



### Project Summary

Breast cancer, a significant health concern, poses life-threatening risks. This project is dedicated to creating a predictive model aimed at early breast cancer diagnosis, a crucial factor in effective treatment and improved patient outcomes.

**Project Objective:**

Our primary goal is to develop a precise predictive model capable of classifying breast tumors as benign or malignant. Through the meticulous analysis of patient data and tumor characteristics, we aim to empower medical professionals with essential information for making well-informed decisions about patient care.

**Dataset:**

We utilize an extensive dataset comprising medical features extracted from breast cancer biopsies. These features encompass critical details such as tumor size, cell shape, and mitotic count. Each record is meticulously labeled as benign or malignant.

**Methodology:**

1. **Data Preprocessing:** Rigorous data preprocessing is conducted, addressing missing values, scaling, and encoding categorical variables for comprehensive analysis.

2. **Model Development:** We employ a range of machine learning algorithms to develop and train predictive models, ensuring robustness and accuracy.

3. **Model Evaluation:** The models are rigorously evaluated using metrics including accuracy, precision, recall, and F1-score to gauge their effectiveness.

**Results and Impact:**

The successful creation of our breast cancer prediction model holds transformative potential for the healthcare landscape. Early diagnosis facilitated by our model can significantly enhance treatments and patient outcomes. This project stands as a pivotal contribution to medical data analysis, emphasizing the value of data-driven approaches in shaping the future of healthcare.

Our dedicated focus on early breast cancer diagnosis exemplifies the power of data-driven methodologies in healthcare. By enabling timely and precise breast cancer diagnoses, our work not only saves lives but also elevates the standard of patient care. This project serves as a testament to the transformative impact of data-driven solutions in addressing critical real-world health challenges.

**Dataset link**: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data

# Problem Statement


Breast cancer, a significant global concern, requires early detection for effective treatment. This project focuses on building a precise predictive model to aid medical professionals in timely and accurate breast cancer diagnosis. Key challenges include addressing data complexity, ensuring high accuracy, and enabling the model's real-world application. The objective is to develop a universally applicable, reliable, and accessible tool for early breast cancer detection, ultimately improving patient care.

# Initiating Exploration!

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
from datetime import datetime as dt


# Import Visualization Libraries
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.impute import KNNImputer
from sklearn.feature_selection import chi2

# Import evaluation metric libraries
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report

# Import preprocessing libraries
from sklearn.preprocessing import StandardScaler

# Import Model
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import RepeatedStratifiedKFold
import xgboost as xgb

# Import model selection libraries
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

# Import warnings
import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
# Load Dataset
# Expects the CSV at ./data/breastcancerdata.csv when running from the repo root
# (download from the Kaggle link above and place it in the data/ folder, since it is
# not committed to this repository per the .gitignore / dataset licensing).
import pandas as pd
df = pd.read_csv('data/breastcancerdata.csv')

### Dataset First View

In [ ]:
# Dataset First Look
# View top 5 rows of the dataset
df.head()

In [ ]:
# View last 5 rows of the dataset
df.tail()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
# Checking number of rows and columns of the dataset using shape
print("Number of rows are: ",df.shape[0])
print("Number of columns are: ",df.shape[1])

### Dataset Information

In [ ]:
# Dataset Info
# Checking information about the dataset using info
df.info()

#### Duplicate Values

In [ ]:
# Checking Dataset Duplicate Value Count
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Checking missing values/null values count for each column
df.isnull().sum()

In [ ]:
# Visualizing the missing values
# Checking Null Value by plotting Heatmap
sns.heatmap(df.isnull(), cbar=False)

### What did you know about your dataset?

There are a total of 33 feature columns where 'diagnosis' is the dependent variable column.

The total number of observations(rows) are 569.

There are no duplicate rows in the dataset.

Also there are missing values in the column 'unnamed: 32'.

## **2. Understanding Your Variables**

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe (all columns included)
df.describe(include= 'all').round(2)

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable using a for loop
for i in df.columns.tolist():
  print("No. of unique values in",i,"is",df[i].nunique())

## **3. Data Wrangling**

### Data Wrangling Code

In [ ]:
# Getting mean columns with diagnosis
m_col = ['diagnosis','radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean']

In [ ]:
# Getting se columns with diagnosis
s_col = ['diagnosis','radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se']

In [ ]:
# Getting worst columns with diagnosis
w_col = ['diagnosis','radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst']

In [ ]:
m_col

In [ ]:
s_col

In [ ]:
w_col

### What all manipulations have you done and insights you found?

Defined the mean columns, se columns and worst columns for ease of plotting graphs.

## **4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables**

#### Chart - 1 : Dependent variable Distribution

In [ ]:
# Chart - 1 Visualization code for Distribution of dependent varaible - diagnosis

# Dependant Column Value Counts
print(df.diagnosis.value_counts())
print(" ")

# Color palette selection
colors = sns.color_palette("Paired")

# Plotting data on chart
plt.figure(figsize=(10,6))
explode = [0,0.1]
textprops = {'fontsize':11}
plt.pie(df['diagnosis'].value_counts(), labels=['Benign (%)','Malignant (%)'], startangle=90, colors=colors, explode = explode, autopct="%1.1f%%",textprops = textprops)
plt.title('Diagnosis (%)', fontsize=15)

# Displaying chart
plt.show()

##### What is/are the insight(s) found from the chart?

From the above chart we come to know that approximately 37.3% of the cases (212 out of 569) indicate the presence of malignant cancer, while the remaining 62.7% (357 out of 569) are classified as benign, indicating the absence of cancer.

#### Chart - 2 : Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
# Create a figure with a specific size for the heatmap
plt.figure(figsize=(20,18))

# Generate a correlation heatmap for the selected data
sns.heatmap(df.corr(),annot=True,linewidths=.5,cmap="YlGnBu")

# Display the heatmap
plt.show()

#### Chart - 3 : Pair Plot

In [ ]:
# Pairplot for mean columns
sns.pairplot(df[m_col], hue='diagnosis', palette='Blues')

# Display Chart
plt.show()

In [ ]:
# Pairplot for se columns
sns.pairplot(df[s_col], hue='diagnosis', palette='Greens')

# Display Chart
plt.show()

In [ ]:
# Pairplot for worst columns
sns.pairplot(df[w_col], hue='diagnosis', palette='Oranges')

# Display Chart
plt.show()

# **5. Feature Engineering & Data Pre-processing**

### 1. Handling Missing Values

#### Unnamed: 32

In [ ]:
# Dropping Unnamed: 32 column
df.drop("Unnamed: 32",axis=1, inplace=True)

### 2. Categorical Encoding

In [ ]:
# Encoding the categorical column
df['diagnosis']=df['diagnosis'].map({'B':0,'M':1})

#### What all categorical encoding techniques i have used & why did i use those techniques?

One hot encoding is used to encode the diagnosis column. Converted malignan as 1 and benign as 0.

### 3. Feature Selection

In [ ]:
# Select features wisely to avoid overfitting
# Dropping id column
df.drop("id",axis=1, inplace=True)

In [ ]:
# Updated columns
df.columns

### 4. Data Splitting

In [ ]:
# Defining the X and y
X = df.drop('diagnosis',axis=1)
y = df['diagnosis']

In [ ]:
# Splitting the data into training and testing sets
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Checking the train distribution of dependent variable
y_train.value_counts()

### 5. Data Scaling

In [ ]:
# Create a StandardScaler object to standardize the data
scaler = StandardScaler()

# Apply the StandardScaler to the training data (X_train) to standardize it
X_train = scaler.fit_transform(X_train)

# Apply the same standardization to the testing data (X_test) to maintain consistency
X_test = scaler.transform(X_test)

 # **6. ML Model Implementation**

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    '''The function will take model, x train, x test, y train, y test
    and then it will fit the model, then make predictions on the trained model,
    it will then print roc-auc score of train and test, then plot the roc, auc curve,
    print confusion matrix for train and test, then print classification report for train and test,
    then plot the feature importances if the model has feature importances,
    and finally it will return the following scores as a list:
    recall_train, recall_test, acc_train, acc_test, roc_auc_train, roc_auc_test, F1_train, F1_test
    '''

    # fit the model on the training data
    model.fit(X_train, y_train)

    # make predictions on the test data
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    pred_prob_train = model.predict_proba(X_train)[:,1]
    pred_prob_test = model.predict_proba(X_test)[:,1]

    # calculate ROC AUC score
    roc_auc_train = roc_auc_score(y_train, y_pred_train)
    roc_auc_test = roc_auc_score(y_test, y_pred_test)
    print("\nTrain ROC AUC:", roc_auc_train)
    print("Test ROC AUC:", roc_auc_test)

    # plot the ROC curve
    fpr_train, tpr_train, thresholds_train = roc_curve(y_train, pred_prob_train)
    fpr_test, tpr_test, thresholds_test = roc_curve(y_test, pred_prob_test)
    plt.plot([0,1],[0,1],'k--')
    plt.plot(fpr_train, tpr_train, label="Train ROC AUC: {:.2f}".format(roc_auc_train))
    plt.plot(fpr_test, tpr_test, label="Test ROC AUC: {:.2f}".format(roc_auc_test))
    plt.legend()
    plt.title("ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.show()

    # calculate confusion matrix
    cm_train = confusion_matrix(y_train, y_pred_train)
    cm_test = confusion_matrix(y_test, y_pred_test)

    fig, ax = plt.subplots(1, 2, figsize=(11,4))

    print("\nConfusion Matrix:")
    sns.heatmap(cm_train, annot=True, xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'], cmap="Oranges", fmt='.4g', ax=ax[0])
    ax[0].set_xlabel("Predicted Label")
    ax[0].set_ylabel("True Label")
    ax[0].set_title("Train Confusion Matrix")

    sns.heatmap(cm_test, annot=True, xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'], cmap="Oranges", fmt='.4g', ax=ax[1])
    ax[1].set_xlabel("Predicted Label")
    ax[1].set_ylabel("True Label")
    ax[1].set_title("Test Confusion Matrix")

    plt.tight_layout()
    plt.show()


    # calculate classification report
    cr_train = classification_report(y_train, y_pred_train, output_dict=True)
    cr_test = classification_report(y_test, y_pred_test, output_dict=True)
    print("\nTrain Classification Report:")
    crt = pd.DataFrame(cr_train).T
    print(crt.to_markdown())
    # sns.heatmap(pd.DataFrame(cr_train).T.iloc[:, :-1], annot=True, cmap="Blues")
    print("\nTest Classification Report:")
    crt2 = pd.DataFrame(cr_test).T
    print(crt2.to_markdown())
    # sns.heatmap(pd.DataFrame(cr_test).T.iloc[:, :-1], annot=True, cmap="Blues")

    precision_train = cr_train['weighted avg']['precision']
    precision_test = cr_test['weighted avg']['precision']

    recall_train = cr_train['weighted avg']['recall']
    recall_test = cr_test['weighted avg']['recall']

    acc_train = accuracy_score(y_true = y_train, y_pred = y_pred_train)
    acc_test = accuracy_score(y_true = y_test, y_pred = y_pred_test)

    F1_train = cr_train['weighted avg']['f1-score']
    F1_test = cr_test['weighted avg']['f1-score']

    model_score = [precision_train, precision_test, recall_train, recall_test, acc_train, acc_test, roc_auc_train, roc_auc_test, F1_train, F1_test ]
    return model_score

In [ ]:
# Create a score dataframe
score = pd.DataFrame(index = ['Precision Train', 'Precision Test','Recall Train','Recall Test','Accuracy Train', 'Accuracy Test','ROC-AUC Train', 'ROC-AUC Test','F1 macro Train', 'F1 macro Test'])

### ML Model - 1 : Logistic regression

In [ ]:
# ML Model - 1 Implementation
lr_model = LogisticRegression(fit_intercept=True, max_iter=10000)

# Model is trained (fit) and predicted in the evaluate model

#### 1. Explaining the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
lr_score = evaluate_model(lr_model, X_train, X_test, y_train, y_test)

In [ ]:
# Updated Evaluation metric Score Chart
score['Logistic regression'] = lr_score
score

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Define the hyperparameter grid
param_grid = {'C': [100,10,1,0.1,0.01,0.001,0.0001],
              'penalty': ['l1', 'l2'],
              'solver':['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']}

# Initializing the logistic regression model
logreg = LogisticRegression(fit_intercept=True, max_iter=10000, random_state=0)

# Repeated stratified kfold
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=4, random_state=0)

# Using GridSearchCV to tune the hyperparameters using cross-validation
grid = GridSearchCV(logreg, param_grid, cv=rskf)
grid.fit(X_train, y_train)

# Select the best hyperparameters found by GridSearchCV
best_params = grid.best_params_
print("Best hyperparameters: ", best_params)

In [ ]:
# Initiate model with best parameters
lr_model2 = LogisticRegression(C=best_params['C'],
                                  penalty=best_params['penalty'],
                                  solver=best_params['solver'],
                                  max_iter=10000, random_state=0)

In [ ]:
# Visualizing evaluation Metric Score chart
lr_score2 = evaluate_model(lr_model2, X_train, X_test, y_train, y_test)

In [ ]:
score['Logistic regression tuned'] = lr_score2

##### Which hyperparameter optimization technique have i used and why?

The hyperparameter optimization technique used is GridSearchCV. GridSearchCV is a method that performs an exhaustive search over a specified parameter grid to find the best hyperparameters for a model. It is a popular method for hyperparameter tuning because it is simple to implement and can be effective in finding good hyperparameters for a model.

The choice of hyperparameter optimization technique depends on various factors such as the size of the parameter space, the computational resources available, and the time constraints. GridSearchCV can be a good choice when the parameter space is relatively small and computational resources are not a major concern.

##### Any improvement in the updated Evaluation metric Score Chart?

In [ ]:
# Updated Evaluation metric Score Chart
score

It appears that hyperparameter tuning improve the performance of the Logistic Regression model on the test set. The tuned Logistic Regression model has higher precision, recall, accuracy, ROC-AUC, and F1 scores on the test set compared to the untuned Logistic Regression model.

### ML Model - 2 : Decision Tree

In [ ]:
# ML Model - 2 Implementation
dt_model = DecisionTreeClassifier(random_state=20)

# Model is trained (fit) and predicted in the evaluate model

#### 1. Explaining the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
dt_score = evaluate_model(dt_model, X_train, X_test, y_train, y_test)

In [ ]:
# Updated Evaluation metric Score Chart
score['Decision Tree'] = dt_score
score

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Define the hyperparameter grid
grid = {'max_depth' : [3,4,5,6,7,8],
        'min_samples_split' : np.arange(2,8),
        'min_samples_leaf' : np.arange(10,20)}

# Initialize the model
model = DecisionTreeClassifier()

# repeated stratified kfold
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=0)

# Initialize GridSearchCV
grid_search = GridSearchCV(model, grid, cv=rskf)

# Fit the GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Select the best hyperparameters
best_params = grid_search.best_params_
print("Best hyperparameters: ", best_params)

In [ ]:
# Train a new model with the best hyperparameters
dt_model2 = DecisionTreeClassifier(max_depth=best_params['max_depth'],
                                 min_samples_leaf=best_params['min_samples_leaf'],
                                 min_samples_split=best_params['min_samples_split'],
                                 random_state=20)

In [ ]:
# Visualizing evaluation Metric Score chart
dt2_score = evaluate_model(dt_model2, X_train, X_test, y_train, y_test)

In [ ]:
score['Decision Tree tuned'] = dt2_score

##### Which hyperparameter optimization technique have i used and why?

The hyperparameter optimization technique used is GridSearchCV. GridSearchCV is a method that performs an exhaustive search over a specified parameter grid to find the best hyperparameters for a model. It is a popular method for hyperparameter tuning because it is simple to implement and can be effective in finding good hyperparameters for a model.

The choice of hyperparameter optimization technique depends on various factors such as the size of the parameter space, the computational resources available, and the time constraints. GridSearchCV can be a good choice when the parameter space is relatively small and computational resources are not a major concern.

##### Any improvement in the updated Evaluation metric Score Chart?

In [ ]:
# Updated Evaluation metric Score Chart
score

It appears that hyperparameter tuning improved the performance of the Decision Tree model on the test set. The tuned Decision Tree model has higher precision, recall, accuracy, ROC-AUC, and F1 scores on the test set compared to the untuned Decision Tree model.

The tuned model is not overfitting like the untuned model.

### ML Model - 3 : Random Forest

In [ ]:
# ML Model - 3 Implementation
rf_model = RandomForestClassifier(random_state=0)

# Model is trained (fit) and predicted in the evaluate model

#### 1. Explaining the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
rf_score = evaluate_model(rf_model, X_train, X_test, y_train, y_test)

In [ ]:
# Updated Evaluation metric Score Chart
score['Random Forest'] = rf_score
score

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Define the hyperparameter grid
grid = {'n_estimators': [10, 50, 100, 200],
              'max_depth': [8, 9, 10, 11, 12,13, 14, 15],
              'min_samples_split': [2, 3, 4, 5]}

# Initialize the model
rf = RandomForestClassifier(random_state=0)

# Repeated stratified kfold
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=0)

# Initialize RandomSearchCV
random_search = RandomizedSearchCV(rf, grid,cv=rskf, n_iter=10, n_jobs=-1)

# Fit the RandomSearchCV to the training data
random_search.fit(X_train, y_train)

# Select the best hyperparameters
best_params = random_search.best_params_
print("Best hyperparameters: ", best_params)

In [ ]:
# Initialize model with best parameters
rf_model2 = RandomForestClassifier(n_estimators = best_params['n_estimators'],
                                 min_samples_leaf= best_params['min_samples_split'],
                                 max_depth = best_params['max_depth'],
                                 random_state=0)

In [ ]:
# Visualizing evaluation Metric Score chart
rf2_score = evaluate_model(rf_model2, X_train, X_test, y_train, y_test)

In [ ]:
score['Random Forest tuned'] = rf2_score

##### Which hyperparameter optimization technique have i used and why?

The hyperparameter optimization technique i used is RandomizedSearchCV. RandomizedSearchCV is a method that performs a random search over a specified parameter grid to find the best hyperparameters for a model. It is a popular method for hyperparameter tuning because it can be more efficient than exhaustive search methods like GridSearchCV when the parameter space is large.

The choice of hyperparameter optimization technique depends on various factors such as the size of the parameter space, the computational resources available, and the time constraints. RandomizedSearchCV can be a good choice when the parameter space is large and computational resources are limited.

##### Any improvement in the updated Evaluation metric Score Chart?

In [ ]:
# Updated Evaluation metric Score Chart
score

It appears that hyperparameter tuning improved the performance of the Random Forest model on the train set. The precision, recall, accuracy, ROC-AUC, and F1 scores on the test set are the same for both the untuned and tuned Random Forest models.

The tuned model is not overfitting like the untuned model.

### ML Model - 4 : SVM (Support Vector Machine)

In [ ]:
# ML Model - 4 Implementation
svm_model = SVC(kernel='linear', random_state=0, probability=True)

# Model is trained (fit) and predicted in the evaluate model

#### 1. Explaining the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:

# Visualizing evaluation Metric Score chart
svm_score = evaluate_model(svm_model, X_train, X_test, y_train, y_test)

In [ ]:
# Updated Evaluation metric Score Chart
score['SVM'] = svm_score
score

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 4 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Define the hyperparameter grid
param_grid = {'C': np.arange(0.1, 10, 0.1),
              'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
              'degree': np.arange(2, 6, 1)}

# Initialize the model
svm = SVC(random_state=0, probability=True)

# Repeated stratified kfold
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=0)

# Initialize RandomizedSearchCV with kfold cross-validation
random_search = RandomizedSearchCV(svm, param_grid, n_iter=10, cv=rskf, n_jobs=-1)

# Fit the RandomizedSearchCV to the training data
random_search.fit(X_train, y_train)

# Select the best hyperparameters
best_params = random_search.best_params_
print("Best hyperparameters: ", best_params)

In [ ]:
# Initialize model with best parameters
svm_model2 = SVC(C = best_params['C'],
           kernel = best_params['kernel'],
           degree = best_params['degree'],
           random_state=0, probability=True)

In [ ]:
# Visualizing evaluation Metric Score chart
svm2_score = evaluate_model(svm_model2, X_train, X_test, y_train, y_test)

In [ ]:
score['SVM tuned'] = svm2_score

##### Which hyperparameter optimization technique have i used and why?

 Here Randomized search is used as a hyperparameter optimization technique.
 Randomized search is a popular technique because it can be more efficient than exhaustive search methods like grid search. Instead of trying all possible combinations of hyperparameters, randomized search samples a random subset of the hyperparameter space. This can save time and computational resources while still finding good hyperparameters for the model.

##### Any improvement in the updated Evaluation metric Score Chart?

In [ ]:
# Updated Evaluation metric Score Chart
score

It appears that hyperparameter tuning improved the performance of the SVM model on the test set. The tuned SVM model has higher precision, recall, accuracy, ROC-AUC score and F1 score on the test set compared to the untuned SVM model.

### ML Model - 5 : Xtreme Gradient Boosting

In [ ]:
# ML Model - 5 Implementation
xgb_model = xgb.XGBClassifier()

# Model is trained (fit) and predicted in the evaluate model

#### 1. Explaining the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
xgb_score = evaluate_model(xgb_model, X_train, X_test, y_train, y_test)

In [ ]:
# Updated Evaluation metric Score Chart
score['XGB'] = xgb_score
score

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 5 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Define the hyperparameter grid
param_grid = {'learning_rate': np.arange(0.01, 0.3, 0.01),
              'max_depth': np.arange(3, 15, 1),
              'n_estimators': np.arange(100, 200, 10)}

# Initialize the model
xgb2 = xgb.XGBClassifier(random_state=0)

# Repeated stratified kfold
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=0)

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(xgb2, param_grid, n_iter=10, cv=rskf)

# Fit the RandomizedSearchCV to the training data
random_search.fit(X_train, y_train)

# Select the best hyperparameters
best_params = random_search.best_params_
print("Best hyperparameters: ", best_params)

In [ ]:
# Initialize model with best parameters
xgb_model2 = xgb.XGBClassifier(learning_rate = best_params['learning_rate'],
                                 max_depth = best_params['max_depth'],
                               n_estimators = best_params['n_estimators'],
                                 random_state=0)

In [ ]:
# Visualizing evaluation Metric Score chart
xgb2_score = evaluate_model(xgb_model2, X_train, X_test, y_train, y_test)

In [ ]:
score['XGB tuned'] = xgb2_score

##### Which hyperparameter optimization technique have i used and why?

Here we have used Randomized search to tune the XGB model.

Randomized search is a popular technique because it can be more efficient than exhaustive search methods like grid search. Instead of trying all possible combinations of hyperparameters, randomized search samples a random subset of the hyperparameter space. This can save time and computational resources while still finding good hyperparameters for the model.

##### Any improvement in the updated Evaluation metric Score Chart?

In [ ]:
# Updated Evaluation metric Score Chart
score

It appears that hyperparameter tuning did not improve the performance of the XGBoost model on the test set. The precision, recall, accuracy, ROC-AUC, and F1 scores on the test set are the same for both the untuned and tuned XGBoost models.

### ML Model - 6 : Naive Bayes

In [ ]:
# ML Model - 6 Implementation
nb_model = GaussianNB()

# Model is trained (fit) and predicted in the evaluate model

#### 1. Explaining the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
nb_score = evaluate_model(nb_model, X_train, X_test, y_train, y_test)

In [ ]:
# Updated Evaluation metric Score Chart
score['Naive Bayes'] = nb_score
score

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 6 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Define the hyperparameter grid
param_grid = {'var_smoothing': np.logspace(0,-9, num=100)}

# Initialize the model
naive = GaussianNB()

# repeated stratified kfold
rskf = RepeatedStratifiedKFold(n_splits=4, n_repeats=4, random_state=0)

# Initialize GridSearchCV
GridSearch = GridSearchCV(naive, param_grid, cv=rskf, n_jobs=-1)

# Fit the GridSearchCV to the training data
GridSearch.fit(X_train, y_train)

# Select the best hyperparameters
best_params = GridSearch.best_params_
print("Best hyperparameters: ", best_params)

In [ ]:
# Initiate model with best parameters
nb_model2 = GaussianNB(var_smoothing = best_params['var_smoothing'])

In [ ]:
# Visualizing evaluation Metric Score chart
nb2_score = evaluate_model(nb_model2, X_train, X_test, y_train, y_test)

In [ ]:
score['Naive Bayes tuned']= nb2_score

##### Which hyperparameter optimization technique have i used and why?

Here we have used the GridSearchCV for optimization of the Naive Bayes model.

GridSearchCV is an exhaustive search method that tries all possible combinations of hyperparameters specified in the hyperparameter grid. This technique can be useful when the number of hyperparameters to tune is small and the range of possible values for each hyperparameter is limited. GridSearchCV can find the best combination of hyperparameters, but it can be computationally expensive for large hyperparameter grids.

##### Any improvement in the updated Evaluation metric Score Chart?

In [ ]:
# Updated Evaluation metric Score Chart
score

It appears that hyperparameter tuning not improved the performance of the Naive Bayes model on the test set. The tuned Naive Bayes model has lower precision, recall, accuracy, ROC-AUC and F1 score on the test set compared to the untuned Naive Bayes model.

### ML Model - 7 : Neural Network

In [ ]:
# ML Model - 7 Implementation
nn_model = MLPClassifier(random_state=0)

# Model is trained (fit) and predicted in the evaluate model

#### 1. Explaining the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
neural_score = evaluate_model(nn_model, X_train, X_test, y_train, y_test)

In [ ]:
# Updated Evaluation metric Score Chart
score['Neural Network'] = neural_score
score

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 7 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Define the hyperparameter grid
param_grid = {'hidden_layer_sizes': np.arange(10, 100, 10),
              'alpha': np.arange(0.0001, 0.01, 0.0001)}

# Initialize the model
neural = MLPClassifier(random_state=0)

# Repeated stratified kfold
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=0)

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(neural, param_grid, n_iter=10, cv=rskf, n_jobs=-1)

# Fit the RandomizedSearchCV to the training data
random_search.fit(X_train, y_train)

# Select the best hyperparameters
best_params = random_search.best_params_
print("Best hyperparameters: ", best_params)

In [ ]:
# Initiate model with best parameters
nn_model2 = MLPClassifier(hidden_layer_sizes = best_params['hidden_layer_sizes'],
                        alpha = best_params['alpha'],
                        random_state = 0)

In [ ]:
# Visualizing evaluation Metric Score chart
neural2_score = evaluate_model(nn_model2, X_train, X_test, y_train, y_test)

In [ ]:
score['Neural Network tuned']= neural2_score

##### Which hyperparameter optimization technique have i used and why?

Here we have used Randomized search to tune the Neural Network model.

Randomized search is a popular technique because it can be more efficient than exhaustive search methods like grid search. Instead of trying all possible combinations of hyperparameters, randomized search samples a random subset of the hyperparameter space. This can save time and computational resources while still finding good hyperparameters for the model.

##### Any improvement in the updated Evaluation metric Score Chart?

In [ ]:
# Updated Evaluation metric Score Chart
score

It appears that hyperparameter tuning did not improve the performance of the neural network model on the test set. The precision, recall, accuracy, ROC-AUC, and F1 scores on the test set are the same for both the untuned and tuned neural network models.

In [ ]:
print(score.to_markdown())

# Selection of best model

In [ ]:
score

In [ ]:
# Removing the overfitted models which have recall, ROC-AUC, f1 scores for train as 1
score_t = score.transpose()            #taking transpose of the score dataframe to create new difference column
remove_models = score_t[score_t['Recall Train']==1].index  #creating a list of models which have 1 for train and score_t['Accuracy Train']==1.0 and score_t['ROC-AUC Train']==1.0 and score_t['F1 macro Train']==1.0
remove_models

adj = score_t.drop(remove_models)         #creating a new dataframe with required models
adj

In [ ]:
def select_best_model(df, metrics):

    best_models = {}
    for metric in metrics:
        max_test = df[metric + ' Test'].max()
        best_model_test = df[df[metric + ' Test'] == max_test].index[0]
        best_model = best_model_test
        best_models[metric] = best_model
    return best_models

In [ ]:
metrics = ['Precision','Recall', 'Accuracy', 'ROC-AUC', 'F1 macro']

best_models = select_best_model(adj, metrics)
print("The best models are:")
for metric, best_model in best_models.items():
    print(f"{metric}: {best_model} - {adj[metric+' Test'][best_model].round(4)}")

## Model Evaluation Summary

The best-performing model in our breast cancer prediction project is **Logistic Regression Tuned**:

- **Precision:** 0.9913
- **Recall:** 0.9912
- **Accuracy:** 0.9912
- **ROC-AUC:** 0.9884
- **F1 Macro:** 0.9912

These scores demonstrate the model's exceptional accuracy, reliability, and balance between precision and recall, making it an ideal choice for breast cancer prediction.


In [ ]:
# Take recall as the primary evaluation metric
score_smpl = score.transpose()
remove_overfitting_models = score_smpl[score_smpl['Recall Train']>=score_smpl['Recall Test']].index
remove_overfitting_models
new_score = score_smpl.drop(remove_overfitting_models)
new_score = new_score.drop(['Precision Train','Precision Test','Accuracy Train','Accuracy Test','ROC-AUC Train','ROC-AUC Test','F1 macro Train','F1 macro Test'], axis=1)
new_score.index.name = 'Classification Model'
print(new_score.to_markdown())

These recall scores demonstrate the models' abilities to identify malignant cases, with Logistic Regression (Tuned) achieving the highest recall rates in both training and test datasets.

### 1. Which evaluation metrics were considered for ensuring a positive business impact, and what was the reasoning behind selecting these specific metrics?

After carefully considering the potential consequences of false positives and false negatives in the context of our business objectives, I have selected recall as the primary evaluation metric for our Breast Cancer prediction model. This means that our goal is to maximize the number of true positives (patients correctly identified as having Breast Cancer) while minimizing the number of false negatives (patients incorrectly identified as not having Breast Cancer). By doing so, we aim to ensure that we correctly identify as many patients with Breast Cancer as possible, even if it means that we may have some false positives.

### 2. Which ML model was selected as the final prediction model from the models created above, and what was the rationale behind this choice?

After evaluating the performance of several machine learning models on the Breast Cancer Wisconsin dataset, I have selected the Logistic regression (tuned) as our final prediction model. This decision was based on the model’s performance on our primary evaluation metric of recall, which measures the ability of the model to correctly identify patients with Breast Cancer. In our analysis, we found that the Logistic regression (tuned) had the highest recall score among the models we evaluated.

We choose recall as our primary evaluation metric because correctly identifying patients with Breast Cancer is critical to achieving our business objectives. By selecting a model with a high recall score, we aim to ensure that we correctly identify as many patients with Breast Cancer as possible, even if it means that we may have some false positives. Overall, we believe that the Logistic regression (tuned) is the best choice for our needs and will help us achieve a positive business impact.

### Conclusion

In the Breast Cancer Prediction Project, our analysis led to vital conclusions:

- **Optimal Model Selection:** After rigorous evaluation, the Tuned Logistic Regression model proved superior among various machine learning models.

- **Robust Model Comparison:** Comparing models like SVM, Decision Tree, Random Forest, Naive Bayes, Neural Network, and XGBoost, Logistic Regression emerged as the standout choice.

- **Primary Metric: Recall:** We prioritized recall, highlighting the model's effectiveness in identifying malignant cases early, crucial for prompt treatment.

- **Healthcare Impact:** Our focus on early, accurate malignant breast cancer detection directly contributes to saving lives and elevating patient care standards.

- **Data-Driven Solutions:** This project showcases the potential of data-driven approaches, emphasizing the importance of model and metric selection in healthcare applications.

In conclusion, our project underscores the pivotal role of model selection and evaluation metrics in healthcare. It demonstrates our dedication to advancing early malignant breast cancer detection, aiming for superior patient outcomes and enhanced healthcare practices.